# General Info

Dataset overview, column types and classes, target distribution and numeric
feature statistics, generated from `configs/config.json`.

In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

DS_ID = os.environ.get("DS_ID", "DS1")

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

config = json.loads((root / "configs" / "config.json").read_text())
ds = config["datasets"]["list"][DS_ID]
base = (root / "configs" / config["paths"]["datasets_dir"] / ds["folder"]).resolve()
file_key = "train" if "train" in ds["files"]["original"] else "data"

df = pd.read_csv(base / ds["files"]["original"][file_key], na_values=ds["na_values"] or None)
target = ds["target"]["column"]

print(f"Dataset: {DS_ID} - {ds['name']}")
print(f"Source:  {ds['source']}")
print(f"Loaded:  {df.shape[0]} rows x {df.shape[1]} cols, target = '{target}'")
df.head()

Dataset: DS1 - UCI Chronic Kidney Disease Dataset
Source:  https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease
Loaded:  400 rows x 25 cols, target = 'class'


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [2]:
def column_type(s):
    n = s.nunique()
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_numeric_dtype(s):
        if n == 2:
            return "numeric (binary)"
        if pd.api.types.is_integer_dtype(s):
            return "numeric (integer)"
        return "numeric (float)"
    if n <= 20 or n / max(len(s), 1) < 0.05:
        return "categorical (string)"
    return "string (free text / id)"


def has_classes(s, max_discrete=10):
    kind = column_type(s)
    if kind in ("boolean", "numeric (binary)") or kind.startswith("categorical"):
        return True
    return kind == "numeric (integer)" and s.nunique() <= max_discrete


def dataset_overview(df, target):
    rows, cols = df.shape
    valid = len(df.dropna())
    lines = [
        "DATASET OVERVIEW",
        f"Instances (rows):            {rows}",
        f"Columns:                     {cols}",
        f"Features (excl. target):     {cols - 1}",
        f"Valid instances (no NaN):    {valid} ({valid / rows:.2%})",
        f"Instances with any NaN:      {rows - valid}",
        f"Duplicate rows:              {df.duplicated().sum()}",
        f"Memory:                      {df.memory_usage(deep=True).sum() / 1024:.1f} KB",
        f"Features per instance ratio: {(cols - 1) / rows:.4f}",
    ]
    return "\n".join(lines)


def column_types_and_classes(df):
    summary = pd.DataFrame({
        "type": {c: column_type(df[c]) for c in df.columns},
        "unique": df.nunique(),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
    })
    summary["classes"] = {
        c: (sorted(df[c].dropna().unique().tolist(), key=str)
            if has_classes(df[c]) else "-")
        for c in df.columns
    }
    counts = summary["type"].value_counts()
    constant = summary.index[summary["unique"] <= 1].tolist()
    lines = [
        "COLUMN TYPES AND CLASSES",
        summary.to_string(),
        "",
        "TYPE COUNTS",
        counts.to_string(),
        f"Constant columns: {constant if constant else 'none'}",
    ]
    return "\n".join(lines)


def target_summary(df, target):
    y = df[target]
    lines = [f"TARGET: {target}"]
    if has_classes(y) or not pd.api.types.is_numeric_dtype(y):
        counts = y.value_counts(dropna=False)
        share = (counts / len(y) * 100).round(2)
        lines += [
            "Task:                  classification",
            f"Number of classes:     {y.nunique()}",
            f"Class values:          {sorted(y.dropna().unique().tolist(), key=str)}",
            pd.DataFrame({"count": counts, "%": share}).to_string(),
            f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}",
            f"Majority-class baseline:   {counts.max() / len(y):.2%}",
        ]
    else:
        lines += ["Task:                  regression", y.describe().to_string()]
    return "\n".join(lines)


def numeric_feature_stats(df, target):
    numeric = df.drop(columns=[target]).select_dtypes(include=[np.number])
    if numeric.empty:
        return "NUMERIC FEATURE STATS\n(none)"
    stats = numeric.describe().T[["min", "max", "mean", "std"]]
    stats["skew"] = numeric.skew()
    return "NUMERIC FEATURE STATS\n" + stats.round(3).to_string()


def full_report(df, target):
    parts = [
        dataset_overview(df, target),
        column_types_and_classes(df),
        target_summary(df, target),
        numeric_feature_stats(df, target),
    ]
    return "\n\n".join(parts)


print(full_report(df, target))

DATASET OVERVIEW
Instances (rows):            400
Columns:                     25
Features (excl. target):     24
Valid instances (no NaN):    158 (39.50%)
Instances with any NaN:      242
Duplicate rows:              0
Memory:                      269.2 KB
Features per instance ratio: 0.0600

COLUMN TYPES AND CLASSES
                       type  unique  missing  missing_%                classes
age         numeric (float)      76        9       2.25                      -
bp          numeric (float)      10       12       3.00                      -
sg          numeric (float)       5       47      11.75                      -
al          numeric (float)       6       46      11.50                      -
su          numeric (float)       6       49      12.25                      -
rbc    categorical (string)       2      152      38.00     [abnormal, normal]
pc     categorical (string)       2       65      16.25     [abnormal, normal]
pcc    categorical (string)       2        4    